# End-to-End Case Study: Predicting and Reducing Customer Churn
### Course template: Machine Learning for Business Insights (Module 9 capstone)

**The business question:** *Which customers are likely to cancel in the next period, why, and what should we do about it?*

**What this notebook does, start to finish**
1. Loads your customer file and checks it (quality, leakage, size)
2. Builds two leakage-safe pipelines: **Logistic Regression** (explainable) and **Random Forest** (flexible)
3. Compares both against a "do nothing" baseline, tunes both, and picks a **champion** with a clear rule
4. Evaluates the champion on a locked test set with **business metrics**: precision, recall, lift, and dollars
5. Chooses **who to contact** using your economics and team capacity
6. Explains **why** customers churn (importance, odds, what-if) and maps each driver to an action
7. Produces **risk tiers**, a **call list**, and a ready-to-edit **business memo**

**How to use it**
- Edit **only the CONFIG cell** (Cell 3): file path, column names, and your business numbers
- Run every cell top to bottom (Runtime → Run all in Colab)
- Leave `CSV_PATH = None` the first time to run on a built-in demo of 3,000 subscription customers

> Demo data is **synthetic**. The workflow is real; the numbers it prints are for practice only.

In [ ]:
# ============================================================
# IMPORTS (all preinstalled in Google Colab; needs scikit-learn 1.2+)
# Local Jupyter: pip install pandas numpy scikit-learn matplotlib seaborn scipy joblib
# ============================================================
import warnings
warnings.filterwarnings("ignore")      # keeps output readable

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import randint
from IPython.display import display, Markdown

from sklearn.model_selection import (
    train_test_split, StratifiedKFold, RepeatedStratifiedKFold,
    cross_validate, cross_val_predict, GridSearchCV, RandomizedSearchCV, learning_curve,
)
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.dummy import DummyClassifier
from sklearn.inspection import permutation_importance
from sklearn.base import clone
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, recall_score, f1_score,
    confusion_matrix, RocCurveDisplay, PrecisionRecallDisplay,
)

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.float_format", lambda x: f"{x:,.3f}")
RANDOM_STATE = 42      # fixes randomness so results repeat exactly
print("Libraries loaded.")

In [ ]:
# ============================================================
# CONFIG: THE ONLY CELL YOU NEED TO EDIT
# ============================================================

# --- 1. Data -------------------------------------------------
CSV_PATH = None                      # e.g. "/content/customers.csv"; None = built-in demo data

# --- 2. Target -----------------------------------------------
TARGET_COLUMN = "churned"            # yes/no column: did the customer cancel?
POSITIVE_VALUE = 1                   # value that means "churned" (1, "Yes", True, "churned", ...)

# --- 3. Inputs -----------------------------------------------
ID_COLUMN = "customer_id"            # used only to label the call list (never as an input)
NUMERIC_COLUMNS = ["tenure_months", "monthly_fee", "logins_30d", "support_tickets_90d",
                   "late_payments_12m", "features_used", "discount_pct", "nps_score"]
CATEGORICAL_COLUMNS = ["plan", "contract_type", "payment_method", "acquisition_channel", "region"]

# Columns to ignore: IDs and LEAKAGE (anything recorded AFTER the scoring date).
# Churn leakage examples: "cancellation_reason", "final_invoice", "account_status",
# "retention_offer_accepted", "days_since_cancel_request".
DROP_COLUMNS = ["customer_id"]

# --- 4. Metric for choosing models ---------------------------
# "average_precision" (PR-AUC) when churn is under ~20%; "roc_auc" otherwise
MAIN_METRIC = "average_precision"

# --- 5. Business economics (rough estimates are fine) ---------
VALUE_IF_SAVED = 900          # value of keeping one customer who would have left (e.g. 12 months of margin)
ACTION_COST = 60              # cost per customer contacted (rep time + offer)
ACTION_SUCCESS_RATE = 0.25    # share of contacted would-be churners the action actually keeps
TEAM_CAPACITY = 400           # max customers your team can contact per cycle (None = no limit)

# --- 6. Driver → action playbook -----------------------------
# Plain-English action for each driver. Used in the recommendations and the call list.
# Add, remove, or reword to match your business. Unlisted drivers get a generic note.
DRIVER_ACTIONS = {
    "contract_type":       "Offer a discounted upgrade from month-to-month to an annual contract",
    "tenure_months":       "Strengthen onboarding: structured check-ins during the first 90 days",
    "logins_30d":          "Trigger a re-engagement sequence and a success-manager call when usage drops",
    "support_tickets_90d": "Route repeat-ticket customers to a senior rep and fix the root cause",
    "late_payments_12m":   "Offer autopay setup and friendly payment reminders",
    "features_used":       "Book a product walkthrough of features the customer isn't using",
    "nps_score":           "Close the loop with detractors within 48 hours",
    "monthly_fee":         "Review plan fit: right-size high-fee, low-usage accounts",
    "payment_method":      "Encourage card or autopay instead of manual invoices",
    "plan":                "Check plan fit and highlight upgrade/downgrade options",
    "discount_pct":        "Review whether discounts are reaching at-risk customers",
}

# --- 7. General ------------------------------------------------
DATA_SIZE_MODE = "auto"       # "auto", "small", or "large"
N_SEARCH_ITER = 25            # random-forest settings to try (raise to 50+ if time allows)
TEST_SIZE = 0.2               # share of customers locked away for the final test
COMPANY_NAME = "Demo Subscription Co."   # used in the memo
print(f"Config set. Target: {TARGET_COLUMN} | Capacity: {TEAM_CAPACITY} | Metric: {MAIN_METRIC}")

In [ ]:
# ============================================================
# LOAD DATA (or build the demo dataset)
# Demo = 3,000 synthetic subscription customers. Don't draw real conclusions.
# ============================================================
def make_demo(n=3_000, seed=RANDOM_STATE):
    rng = np.random.default_rng(seed)
    tenure = rng.integers(1, 73, n)
    contract = rng.choice(["month-to-month", "annual", "two-year"], n, p=[0.55, 0.30, 0.15])
    plan = rng.choice(["basic", "standard", "premium"], n, p=[0.4, 0.4, 0.2])
    fee = (np.select([plan == "basic", plan == "standard"], [29, 59], 99) + rng.normal(0, 5, n)).round(2)
    logins = rng.poisson(14, n).astype(float)
    tickets = rng.poisson(1.1, n)
    late = rng.poisson(0.35, n)
    features = rng.binomial(10, 0.45, n)
    discount = rng.choice([0, 0, 0, 10, 15, 25], n)
    nps = np.clip(rng.normal(7.5, 2, n), 0, 10).round()
    payment = rng.choice(["card", "bank_transfer", "invoice"], n, p=[0.5, 0.3, 0.2])
    channel = rng.choice(["organic", "paid_search", "referral", "partner"], n, p=[0.4, 0.3, 0.2, 0.1])
    region = rng.choice(["northeast", "south", "midwest", "west"], n)
    # Hidden "real world" churn logic, including one interaction (frustrated AND disengaged)
    z = (-0.5 + 1.5 * (contract == "month-to-month") - 0.6 * (contract == "two-year")
         - 0.035 * tenure - 0.06 * np.minimum(logins, 25) + 0.3 * np.minimum(tickets, 5)
         + 0.6 * late - 0.12 * features - 0.18 * (nps - 7) + 0.012 * (fee - 55)
         + 0.5 * (payment == "invoice") + 1.0 * ((tickets >= 3) & (logins < 9)) - 0.02 * discount)
    churn = (rng.random(n) < 1 / (1 + np.exp(-z))).astype(int)
    df = pd.DataFrame({
        "customer_id": [f"CUST-{i:05d}" for i in range(n)],
        "tenure_months": tenure, "monthly_fee": fee, "logins_30d": logins,
        "support_tickets_90d": tickets, "late_payments_12m": late, "features_used": features,
        "discount_pct": discount, "nps_score": nps, "plan": plan, "contract_type": contract,
        "payment_method": payment, "acquisition_channel": channel, "region": region, "churned": churn,
    })
    df.loc[rng.choice(n, 60, replace=False), "logins_30d"] = np.nan      # realistic blanks
    df.loc[rng.choice(n, 150, replace=False), "nps_score"] = np.nan      # not everyone answers surveys
    df.loc[rng.choice(n, 25, replace=False), "acquisition_channel"] = np.nan
    return df

if CSV_PATH is None:
    df = make_demo()
    print("Using DEMO data (synthetic). Set CSV_PATH in CONFIG to use your own file.")
else:
    df = pd.read_csv(CSV_PATH)
    print(f"Loaded {CSV_PATH}")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
df.head()

In [ ]:
# ============================================================
# CONFIG CHECK + TARGET (_y = 1 churned, 0 stayed)
# ============================================================
need = [TARGET_COLUMN] + NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
missing = [c for c in need if c not in df.columns]
if missing:
    raise ValueError(f"Columns in CONFIG but not in your data: {missing}\nYour columns: {list(df.columns)}")
overlap = set(DROP_COLUMNS + [ID_COLUMN]) & set(NUMERIC_COLUMNS + CATEGORICAL_COLUMNS)
if overlap:
    raise ValueError(f"These are listed as inputs AND as ID/dropped columns: {overlap}")
if ID_COLUMN and ID_COLUMN not in df.columns:
    print(f"Note: ID_COLUMN '{ID_COLUMN}' not found; the call list will use row numbers.")
    ID_COLUMN = None

# Clean numbers stored as text ("$49.00", "12%") and store numerics as decimals
for c in NUMERIC_COLUMNS:
    if not pd.api.types.is_numeric_dtype(df[c]):
        df[c] = pd.to_numeric(df[c].astype(str).str.replace(r"[$,%\s]", "", regex=True), errors="coerce")
        print(f"Converted '{c}' from text to numbers.")
df[NUMERIC_COLUMNS] = df[NUMERIC_COLUMNS].astype(float)

df = df.dropna(subset=[TARGET_COLUMN]).reset_index(drop=True)
print("Target values found:", df[TARGET_COLUMN].value_counts().to_dict())
df["_y"] = (df[TARGET_COLUMN].astype(str).str.strip().str.lower()
            == str(POSITIVE_VALUE).strip().lower()).astype(int)
if df["_y"].nunique() < 2:
    raise ValueError("Only one class after mapping. Check POSITIVE_VALUE.")
CHURN_RATE = df["_y"].mean()
print(f"Churn rate: {CHURN_RATE:.1%} ({df['_y'].sum():,} of {len(df):,} customers)")

## Section 1: Light EDA: Quality, Leakage, and Size

Three quick checks before any modeling:
1. **Quality:** blanks, duplicates, odd values, category typos
2. **Leakage:** any single input that predicts churn *too* well was probably recorded after the customer left
3. **Size:** how many churners we have decides how cautious the rest of the notebook should be

In [ ]:
# ============================================================
# DATA QUALITY + FIRST BUSINESS PICTURE
# ============================================================
m = df[NUMERIC_COLUMNS + CATEGORICAL_COLUMNS].isna().sum()
print("Missing values:\n", (m[m > 0].to_string() if m.sum() else "None"))
print("Duplicate rows:", df.duplicated().sum())
if ID_COLUMN:
    print("Duplicate customer IDs:", df[ID_COLUMN].duplicated().sum())
display(df[NUMERIC_COLUMNS].describe().T)

# Numeric inputs: average for churners vs. stayers (the "pivot table first" habit)
print("Average value: churned vs. stayed")
display(df.groupby("_y")[NUMERIC_COLUMNS].mean().T.rename(columns={0: "stayed", 1: "churned"}))

# Categorical inputs: churn rate by group
fig, axes = plt.subplots(1, len(CATEGORICAL_COLUMNS), figsize=(4 * len(CATEGORICAL_COLUMNS), 3.2))
for ax, c in zip(np.atleast_1d(axes), CATEGORICAL_COLUMNS):
    df.groupby(c)["_y"].mean().sort_values().plot.barh(ax=ax, color="indianred")
    ax.axvline(CHURN_RATE, color="black", ls="--", lw=1)
    ax.set_title(f"Churn rate by {c}"); ax.set_xlabel(""); ax.set_ylabel("")
plt.tight_layout(); plt.show()
for c in CATEGORICAL_COLUMNS:
    rare = df[c].value_counts()[lambda s: s < 30]
    if len(rare):
        print(f"⚠ {c}: rare values {list(rare.index)} (<30 customers). Consider grouping into 'Other'.")

In [ ]:
# ============================================================
# LEAKAGE SNIFF TEST: how well does each numeric input ALONE separate churners?
# ROC-AUC 0.5 = useless, 1.0 = perfect. One input above 0.90 is a red flag.
# ============================================================
print("Single-input ROC-AUC:")
for c in NUMERIC_COLUMNS:
    s = df[[c, "_y"]].dropna()
    auc = roc_auc_score(s["_y"], s[c]); auc = max(auc, 1 - auc)
    print(f"  {c:<22} {auc:.2f}" + ("  ⚠ suspiciously strong: is it known BEFORE churn?" if auc > 0.90 else ""))
for c in CATEGORICAL_COLUMNS:
    rates = df.groupby(c)["_y"].mean()
    if ((rates > 0.97) | (rates < 0.03)).any() and df[c].nunique() > 1:
        print(f"  ⚠ {c}: some groups churn almost always or never. Check for leakage (e.g. a 'status' column).")

# ============================================================
# DATA SIZE CHECK: what matters is the number of CHURNERS
# ============================================================
n_pos = int(df["_y"].sum()); n_min = min(n_pos, len(df) - n_pos)
n_feat = len(NUMERIC_COLUMNS) + sum(df[c].nunique() - 1 for c in CATEGORICAL_COLUMNS)
MODE = ("small" if n_min < 200 else "large") if DATA_SIZE_MODE == "auto" else DATA_SIZE_MODE
print(f"\nCustomers {len(df):,} | Churners {n_pos:,} | Features after encoding ~{n_feat} | "
      f"Churners per feature {n_min / n_feat:.1f} | Mode: {MODE.upper()}")
if MODE == "small":
    print("SMALL-DATA PATH: repeated CV, cautious settings, and logistic regression favored unless the forest clearly wins.")
if n_min / n_feat < 10:
    print("⚠ Fewer than 10 churners per feature: trim inputs to those with a clear business reason.")

## Section 2: Split and Validation Plan

- **Test set (20%)**: locked away until the end. Used **once** for the numbers we report.
- **Cross-validation** on the other 80%: used to compare and tune models. Stratified, so every fold has the same churn rate.
- Small data uses **repeated** CV so one lucky split can't decide anything.

> **Time note:** if your file spans many months, the most honest test is *train on older customers,
> test on the most recent period*. A random split is fine for a first pass.

In [ ]:
# ============================================================
# SPLIT + CV
# ============================================================
feature_cols = NUMERIC_COLUMNS + CATEGORICAL_COLUMNS
X, y = df[feature_cols], df["_y"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

CV = (RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=RANDOM_STATE) if MODE == "small"
      else StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE))
N_FOLDS = CV.get_n_splits()
# Out-of-fold predictions need each row predicted exactly once → plain 5-fold
OOF_CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
print(f"Train {len(X_train):,} ({y_train.mean():.1%} churn) | Test {len(X_test):,} ({y_test.mean():.1%} churn) | "
      f"CV: {type(CV).__name__} ({N_FOLDS} scores per model)")

## Section 3: Two Leakage-Safe Pipelines

| | Logistic Regression | Random Forest |
|---|---|---|
| Numbers | Fill blanks (median) → **scale** | Fill blanks (median), no scaling needed |
| Categories | Fill blanks → **one-hot** (reference = most common value) | Fill blanks → **ordinal codes** |
| Strength | Clear, explainable effects (odds) | Captures curves and interactions |

Every step lives **inside** the pipeline, so during cross-validation the fill values, scaling, and encoding
are learned from training folds only. That's what "leakage-safe" means.

**Why no `class_weight="balanced"` here?** It inflates probabilities for churners. We want probabilities that
read as *real risk* for the memo and risk tiers, and we handle the imbalance with a business-driven threshold instead.

In [ ]:
# ============================================================
# PIPELINES
# ============================================================
REFERENCE_GROUPS = [X_train[c].mode().iloc[0] for c in CATEGORICAL_COLUMNS]   # most common value per category

logit_prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), NUMERIC_COLUMNS),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("onehot", OneHotEncoder(drop=REFERENCE_GROUPS, handle_unknown="ignore", sparse_output=False))]),
     CATEGORICAL_COLUMNS),
])
rf_prep = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERIC_COLUMNS),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="constant", fill_value="Unknown")),
                      ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1))]),
     CATEGORICAL_COLUMNS),
], verbose_feature_names_out=False)

def make_logit(**params):
    return Pipeline([("prep", logit_prep), ("model", LogisticRegression(max_iter=5_000, **params))])

def make_rf(**params):
    return Pipeline([("prep", rf_prep),
                     ("model", RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1, **params))])

print("Reference groups:", {c: str(r) for c, r in zip(CATEGORICAL_COLUMNS, REFERENCE_GROUPS)})
print("Pipelines ready.")

## Section 4: Baselines

- **Dummy:** predicts the overall churn rate for everyone. Zero skill; every model must beat it.
- **Logistic Regression (default)** and **Random Forest (default)**: out-of-the-box settings.

We show training and validation scores: a big gap = memorizing, not learning.

In [ ]:
# ============================================================
# BASELINE COMPARISON (cross-validated, training data only)
# ============================================================
SCORING = {"ROC-AUC": "roc_auc", "PR-AUC": "average_precision"}
KEY = {v: k for k, v in SCORING.items()}.get(MAIN_METRIC, MAIN_METRIC)
if MAIN_METRIC not in SCORING.values():
    SCORING[KEY] = MAIN_METRIC

def cv_summary(model, name):
    r = cross_validate(model, X_train, y_train, cv=CV, scoring=SCORING, return_train_score=True, n_jobs=1)
    row = {"model": name}
    for k in SCORING:
        row[k] = r[f"test_{k}"].mean(); row[f"{k} ±"] = r[f"test_{k}"].std(); row[f"{k} (train)"] = r[f"train_{k}"].mean()
    return row

res = pd.DataFrame([
    cv_summary(Pipeline([("prep", rf_prep), ("model", DummyClassifier(strategy="prior"))]), "Baseline (dummy)"),
    cv_summary(make_logit(), "Logistic (default)"),
    cv_summary(make_rf(), "Random forest (default)"),
]).set_index("model")
display(res)
print(f"""
HOW TO READ THIS
- PR-AUC of random guessing = churn rate ({y_train.mean():.2f}). ROC-AUC of random guessing = 0.50.
- ± = spread across folds. Differences smaller than ± are ties.
- A default forest usually scores ~1.00 on training data: it memorizes customers. Judge it on validation only.""")

## Section 5: Tune Both Models, Then Pick a Champion

**Logistic Regression → GridSearchCV** over `C` (regularization strength). Small grid, so we try every value.
Simplicity rule: the **smallest C** (most cautious) within one standard error of the best.

**Random Forest → RandomizedSearchCV** over tree depth, leaf size, features per split, and sampling.
Simplicity rule: the **least overfit** setting (smallest train–validation gap) within one standard error of the best.

**Champion rule (decided before we look at the test set):**
the forest must beat logistic regression by **more than the fold-to-fold noise** (±).
Otherwise we keep logistic regression: equal accuracy, much easier to explain and maintain.

In [ ]:
# ============================================================
# TUNE LOGISTIC REGRESSION (grid search over C)
# ============================================================
C_grid = [0.01, 0.03, 0.1, 0.3, 1, 3] if MODE == "small" else [0.01, 0.03, 0.1, 0.3, 1, 3, 10, 30]
gs = GridSearchCV(make_logit(), {"model__C": C_grid}, cv=CV, scoring=MAIN_METRIC, n_jobs=-1, return_train_score=True)
gs.fit(X_train, y_train)

lr_tbl = pd.DataFrame({"C": C_grid, "valid": gs.cv_results_["mean_test_score"],
                       "±": gs.cv_results_["std_test_score"], "train": gs.cv_results_["mean_train_score"]})
display(lr_tbl)
best = lr_tbl.loc[lr_tbl["valid"].idxmax()]
se = best["±"] / np.sqrt(N_FOLDS)
BEST_C = float(lr_tbl.loc[lr_tbl["valid"] >= best["valid"] - se, "C"].min())
print(f"Best C {best['C']:g} ({KEY} {best['valid']:.3f}); CHOSEN C = {BEST_C:g} (most cautious within one standard error, ±{se:.3f})")
logit_tuned = make_logit(C=BEST_C)

In [ ]:
# ============================================================
# TUNE RANDOM FOREST (randomized search)
# ============================================================
if MODE == "small":
    space = {"model__n_estimators": [300],
             "model__max_depth": [3, 4, 5, 6, 8],
             "model__min_samples_leaf": randint(10, 40),
             "model__max_features": ["sqrt", 0.3, 0.5]}
else:
    space = {"model__n_estimators": [300],
             "model__max_depth": [None, 6, 8, 12, 16],
             "model__min_samples_leaf": randint(2, 60),
             "model__max_features": ["sqrt", 0.3, 0.5, 0.7],
             "model__max_samples": [None, 0.5, 0.7]}

rs = RandomizedSearchCV(make_rf(), space, n_iter=N_SEARCH_ITER, cv=CV, scoring=MAIN_METRIC,
                        random_state=RANDOM_STATE, n_jobs=-1, return_train_score=True)
rs.fit(X_train, y_train)

cvr = rs.cv_results_
rf_tbl = pd.DataFrame(cvr["params"]).rename(columns=lambda c: c.replace("model__", ""))
rf_tbl["valid"], rf_tbl["±"], rf_tbl["train"] = cvr["mean_test_score"], cvr["std_test_score"], cvr["mean_train_score"]
rf_tbl["gap"] = rf_tbl["train"] - rf_tbl["valid"]
rf_tbl = rf_tbl.sort_values("valid", ascending=False, kind="stable")      # index still points to cv_results_
display(rf_tbl.head(8))

best = rf_tbl.iloc[0]
se = best["±"] / np.sqrt(N_FOLDS)
within = rf_tbl[rf_tbl["valid"] >= best["valid"] - se]
RF_PARAMS = {k.replace("model__", ""): v for k, v in cvr["params"][within["gap"].idxmin()].items()}
print(f"Best {KEY} {best['valid']:.3f}; CHOSEN (least overfit within one standard error): {RF_PARAMS}")
rf_tuned = make_rf(**RF_PARAMS)

In [ ]:
# ============================================================
# CHAMPION RULE
# ============================================================
res2 = pd.concat([res, pd.DataFrame([cv_summary(logit_tuned, "Logistic (tuned)"),
                                      cv_summary(rf_tuned, "Random forest (tuned)")]).set_index("model")])
display(res2[[KEY, f"{KEY} ±", f"{KEY} (train)", "ROC-AUC"]])

lr_score, rf_score = res2.loc["Logistic (tuned)", KEY], res2.loc["Random forest (tuned)", KEY]
noise = res2.loc["Random forest (tuned)", f"{KEY} ±"]
if rf_score - lr_score > noise:
    CHAMPION_NAME, champion = "Random forest", rf_tuned
    print(f"🏆 CHAMPION: Random forest. It beats logistic by {rf_score - lr_score:.3f} {KEY}, more than the noise (±{noise:.3f}).")
else:
    CHAMPION_NAME, champion = "Logistic regression", logit_tuned
    print(f"🏆 CHAMPION: Logistic regression. The forest's edge ({rf_score - lr_score:+.3f} {KEY}) is within the noise "
          f"(±{noise:.3f}), so the simpler, explainable model wins.")

## Section 6: Final Test-Set Evaluation

The champion is fit on all training customers and scored on the locked test set **once**.
We show the runner-up too, for transparency, but the decision was already made in Section 5.

In [ ]:
# ============================================================
# TEST SET
# ============================================================
champion.fit(X_train, y_train)
proba_test = champion.predict_proba(X_test)[:, 1]
runner_up = (rf_tuned if CHAMPION_NAME == "Logistic regression" else logit_tuned).fit(X_train, y_train)
proba_runner = runner_up.predict_proba(X_test)[:, 1]

summary = pd.DataFrame({
    f"{CHAMPION_NAME} (champion)": [roc_auc_score(y_test, proba_test), average_precision_score(y_test, proba_test)],
    "Runner-up": [roc_auc_score(y_test, proba_runner), average_precision_score(y_test, proba_runner)],
    "Random guessing": [0.5, y_test.mean()],
}, index=["ROC-AUC", "PR-AUC"])
display(summary)
TEST_ROC, TEST_PR = summary.iloc[0, 0], summary.iloc[1, 0]

fig, ax = plt.subplots(1, 2, figsize=(12, 4.5))
RocCurveDisplay.from_predictions(y_test, proba_test, ax=ax[0], name=f"{CHAMPION_NAME} (champion)")
RocCurveDisplay.from_predictions(y_test, proba_runner, ax=ax[0], name="Runner-up")
ax[0].plot([0, 1], [0, 1], "k--", lw=0.8); ax[0].set_title("ROC curve (test)")
PrecisionRecallDisplay.from_predictions(y_test, proba_test, ax=ax[1], name=f"{CHAMPION_NAME} (champion)")
PrecisionRecallDisplay.from_predictions(y_test, proba_runner, ax=ax[1], name="Runner-up")
ax[1].axhline(y_test.mean(), color="k", ls="--", lw=0.8); ax[1].set_title("Precision–recall (test)")
plt.tight_layout(); plt.show()

## Section 7: Who Should We Contact? (Business Threshold)

**Expected value = churners reached × VALUE_IF_SAVED × ACTION_SUCCESS_RATE − customers contacted × ACTION_COST**

- Chosen on **out-of-fold** predictions (each training customer scored by a model that never saw them)
- Limited by **TEAM_CAPACITY**
- Scaled to your full customer file, so it reads as "per campaign cycle"
- Then checked **once** on the test set

In [ ]:
# ============================================================
# OUT-OF-FOLD THRESHOLD TABLE
# ============================================================
proba_oof = cross_val_predict(champion, X_train, y_train, cv=OOF_CV, method="predict_proba")[:, 1]
CUSTOMER_BASE = len(df)
BREAK_EVEN = ACTION_COST / (VALUE_IF_SAVED * ACTION_SUCCESS_RATE)

def threshold_table(y_true, proba, value=VALUE_IF_SAVED, cost=ACTION_COST, success=ACTION_SUCCESS_RATE):
    y_true = np.asarray(y_true)
    cands = list(np.round(np.arange(0.05, 0.951, 0.025), 3))
    if TEAM_CAPACITY:
        cands.append(float(np.quantile(proba, 1 - min(TEAM_CAPACITY / CUSTOMER_BASE, 1.0))))
    scale = CUSTOMER_BASE / len(y_true)
    rows = []
    for t in sorted(set(cands)):
        f = proba >= t; nf, tp = int(f.sum()), int((f & (y_true == 1)).sum())
        rows.append({"threshold": t, "contacted": round(nf * scale), "churners reached": round(tp * scale),
                     "precision": tp / nf if nf else np.nan, "recall": tp / max(y_true.sum(), 1),
                     "expected value": (tp * value * success - nf * cost) * scale,
                     "within capacity": (TEAM_CAPACITY is None) or (nf * scale <= TEAM_CAPACITY * 1.001)})
    return pd.DataFrame(rows)

tt = threshold_table(y_train, proba_oof)
ok = tt[tt["within capacity"]]
best_row = ok.loc[ok["expected value"].idxmax()]
THRESHOLD = float(best_row["threshold"])
uncapped = tt.loc[tt["expected value"].idxmax()]

display(tt.iloc[::2].style.format({"threshold": "{:.3f}", "precision": "{:.2f}", "recall": "{:.2f}",
                                    "contacted": "{:,.0f}", "churners reached": "{:,.0f}",
                                    "expected value": "${:,.0f}"}).hide(axis="index"))
print(f"""Break-even precision: {BREAK_EVEN:.1%}
CHOSEN THRESHOLD: {THRESHOLD:.3f} → contact ~{best_row['contacted']:,.0f} customers per cycle,
reaching {best_row['recall']:.0%} of churners at {best_row['precision']:.0%} precision. Estimated value ≈ ${best_row['expected value']:,.0f} per cycle.""")
if TEAM_CAPACITY and uncapped["expected value"] > best_row["expected value"] * 1.05:
    print(f"💡 Without the capacity limit, the best plan would contact ~{uncapped['contacted']:,.0f} customers for "
          f"≈ ${uncapped['expected value']:,.0f}. Extra capacity is worth ≈ ${uncapped['expected value'] - best_row['expected value']:,.0f} per cycle.")

tt_test = threshold_table(y_test, proba_test)
rt = tt_test.iloc[(tt_test["threshold"] - THRESHOLD).abs().argmin()]
TEST_PRECISION, TEST_RECALL, TEST_VALUE = rt["precision"], rt["recall"], rt["expected value"]
print(f"TEST SET at {THRESHOLD:.3f}: precision {TEST_PRECISION:.0%}, recall {TEST_RECALL:.0%}, ≈ ${TEST_VALUE:,.0f} per cycle")

# Sensitivity: does the program still pay if our estimates are off?
sens = []
for v in [0.5, 1.0, 1.5]:
    for s in [0.5, 1.0, 1.5]:
        t_ = threshold_table(y_train, proba_oof, value=VALUE_IF_SAVED * v, success=min(ACTION_SUCCESS_RATE * s, 1))
        t_ = t_[t_["within capacity"]]; b = t_.loc[t_["expected value"].idxmax()]
        sens.append({"value if saved": VALUE_IF_SAVED * v, "success rate": min(ACTION_SUCCESS_RATE * s, 1),
                     "best threshold": b["threshold"], "expected value": b["expected value"]})
sens = pd.DataFrame(sens)
display(sens.style.format({"value if saved": "${:,.0f}", "success rate": "{:.0%}", "best threshold": "{:.3f}",
                           "expected value": "${:,.0f}"}).hide(axis="index"))
POSITIVE_SCENARIOS = int((sens["expected value"] > 0).sum())
print(f"Program value is positive in {POSITIVE_SCENARIOS} of 9 scenarios.")

In [ ]:
# ============================================================
# LIFT + CUMULATIVE GAINS (test set)
# ============================================================
lift = pd.DataFrame({"y": y_test.values, "p": proba_test})
lift["decile"] = pd.qcut(lift["p"].rank(method="first", ascending=False), 10, labels=range(1, 11))
lt = lift.groupby("decile", observed=True).agg(customers=("y", "size"), churners=("y", "sum"), avg_risk=("p", "mean"))
lt["churn rate"] = lt["churners"] / lt["customers"]
lt["lift"] = lt["churn rate"] / lift["y"].mean()
lt["cumulative % of churners"] = lt["churners"].cumsum() / lt["churners"].sum() * 100
display(lt.style.format({"avg_risk": "{:.2f}", "churn rate": "{:.1%}", "lift": "{:.1f}x", "cumulative % of churners": "{:.0f}%"}))
TOP10_LIFT, TOP20_CAPTURE = lt["lift"].iloc[0], lt["cumulative % of churners"].iloc[1]

plt.figure(figsize=(6, 4))
plt.plot(np.arange(0, 110, 10), np.r_[0, lt["cumulative % of churners"].values], "o-", label="model")
plt.plot([0, 100], [0, 100], "k--", label="random")
plt.xlabel("% of customers contacted (highest risk first)"); plt.ylabel("% of churners reached")
plt.title("Cumulative gains (test)"); plt.legend(); plt.show()
print(f"Top 10% riskiest customers churn at {TOP10_LIFT:.1f}x the average rate; "
      f"contacting the top 20% reaches {TOP20_CAPTURE:.0f}% of churners.")

## Section 8: Why Do Customers Churn?

Three complementary views:
1. **Permutation importance** (champion, test set): which inputs the model relies on most
2. **Odds multipliers** (tuned logistic): direction and size in plain words, e.g. "each extra support ticket raises churn odds by 30%"
3. **What-if** (champion): nudge each driver for every test customer and watch average risk move

⚠️ These describe patterns in historical data, not proven cause and effect. Treat them as strong leads
and confirm with a controlled test (see the pilot plan in Section 9).

In [ ]:
# ============================================================
# PERMUTATION IMPORTANCE (champion, test set)
# ============================================================
perm = permutation_importance(champion, X_test, y_test, scoring=MAIN_METRIC, n_repeats=15,
                              random_state=RANDOM_STATE, n_jobs=-1)
imp = pd.DataFrame({"importance": perm.importances_mean, "±": perm.importances_std},
                   index=feature_cols).sort_values("importance", ascending=False)
imp["rank"] = range(1, len(imp) + 1)
display(imp)
imp["importance"][::-1].plot.barh(xerr=imp["±"][::-1], figsize=(7, 0.45 * len(imp) + 1.5), color="steelblue")
plt.title(f"What the {CHAMPION_NAME.lower()} relies on (drop in {KEY} when shuffled)"); plt.tight_layout(); plt.show()

In [ ]:
# ============================================================
# ODDS MULTIPLIERS (from the tuned logistic model, always fitted for interpretation)
# ============================================================
logit_fit = clone(logit_tuned).fit(X_train, y_train)
prep = logit_fit.named_steps["prep"]
coefs = logit_fit.named_steps["model"].coef_[0]
names = [n.split("__", 1)[1] for n in prep.get_feature_names_out()]
scaler = prep.named_transformers_["num"].named_steps["scale"]
SD = dict(zip(NUMERIC_COLUMNS, scaler.scale_))

odds_rows = []
for i, (name, c) in enumerate(zip(names, coefs)):
    if i < len(NUMERIC_COLUMNS):
        odds_rows.append({"driver": name, "source column": name, "type": "numeric",
                          "odds × per +1 unit": np.exp(c / SD[name]), "odds × per typical change": np.exp(c),
                          "typical change": SD[name]})
    else:
        src = next(col for col in CATEGORICAL_COLUMNS if name.startswith(col + "_"))
        odds_rows.append({"driver": name, "source column": src, "type": "category vs. reference",
                          "odds × per +1 unit": np.exp(c), "odds × per typical change": np.exp(c), "typical change": np.nan})
odds = pd.DataFrame(odds_rows)
odds = odds.reindex(odds["odds × per typical change"].map(lambda v: abs(np.log(v))).sort_values(ascending=False).index)
display(odds.drop(columns="source column").style.format({"odds × per +1 unit": "{:.2f}", "odds × per typical change": "{:.2f}",
                                                         "typical change": "{:,.1f}"}, na_rep="").hide(axis="index"))
# Direction per numeric column (+1 = higher value → more churn), used for per-customer reasons later
DIRECTION = {r["driver"]: (1 if r["odds × per +1 unit"] > 1 else -1) for _, r in odds.iterrows() if r["type"] == "numeric"}

In [ ]:
# ============================================================
# WHAT-IF (champion): average churn risk if a driver moves
# Numeric: everyone +1 typical change. Category: everyone set to each value.
# ============================================================
base_risk = proba_test.mean()
wi = []
for c in NUMERIC_COLUMNS:
    Xw = X_test.copy(); Xw[c] = Xw[c] + SD[c]
    wi.append({"driver": c, "what-if": f"+{SD[c]:,.1f} (typical change)",
               "avg risk change (pts)": (champion.predict_proba(Xw)[:, 1].mean() - base_risk) * 100})
for c in CATEGORICAL_COLUMNS:
    risks = {}
    for v in X_train[c].dropna().unique():
        Xw = X_test.copy(); Xw[c] = v
        risks[v] = champion.predict_proba(Xw)[:, 1].mean()
    lo, hi = min(risks, key=risks.get), max(risks, key=risks.get)
    wi.append({"driver": c, "what-if": f"everyone '{hi}' vs. everyone '{lo}'",
               "avg risk change (pts)": (risks[hi] - risks[lo]) * 100})
whatif = pd.DataFrame(wi).set_index("driver")

# ============================================================
# DRIVER → ACTION TABLE (the bridge from model to business)
# ============================================================
def direction_text(col):
    if col in DIRECTION:
        return "higher → more churn" if DIRECTION[col] > 0 else "higher → less churn"
    w = whatif.loc[col, "what-if"]
    return "riskiest: " + w.split("'")[1]

drivers = imp[["rank", "importance"]].join(whatif).reset_index().rename(columns={"index": "driver"})
drivers["direction"] = drivers["driver"].map(direction_text)
drivers["recommended action"] = drivers["driver"].map(lambda c: DRIVER_ACTIONS.get(c, "Investigate with the business owner"))
drivers["actionable now"] = drivers["importance"] > imp["±"].reindex(drivers["driver"]).values
display(drivers[["rank", "driver", "direction", "what-if", "avg risk change (pts)", "recommended action"]]
        .style.format({"avg risk change (pts)": "{:+.1f}"}).hide(axis="index"))
TOP_DRIVERS = drivers[drivers["actionable now"]].head(5)

print("PLAIN-ENGLISH INSIGHTS:")
for _, r in TOP_DRIVERS.iterrows():
    print(f"- {r['driver']}: {r['direction']}; {r['what-if']} shifts average risk by {r['avg risk change (pts)']:+.1f} points. "
          f"→ {r['recommended action']}.")

## Section 9: From Model to Action

This section turns scores into a plan:
1. **Risk tiers:** who gets a call, who gets a lighter touch, who gets nothing
2. **Call list:** each high-risk customer with their top risk reasons and a suggested action
3. **Business memo:** a summary you can edit and send
4. **Pilot plan:** how to prove the program works before scaling it

> **Honest scores:** to avoid flattering ourselves, every customer below is scored by a model that did **not**
> train on them (out-of-fold for training customers, the trained champion for test customers).
> In real use, refit on all history and score your **current active customers** instead.

In [ ]:
# ============================================================
# RISK TIERS
# ============================================================
scores = pd.Series(np.nan, index=df.index)
scores.loc[X_train.index] = proba_oof
scores.loc[X_test.index] = proba_test

scored = df[[ID_COLUMN] if ID_COLUMN else []].copy()
scored["churn_risk"] = scores
scored["actual_churned"] = df["_y"]           # known here because this is historical data

ranked = scored["churn_risk"].rank(ascending=False, method="first")
high = scored["churn_risk"] >= THRESHOLD
if TEAM_CAPACITY:
    high &= ranked <= TEAM_CAPACITY
medium = ~high & (scored["churn_risk"] >= CHURN_RATE)
scored["tier"] = np.select([high, medium], ["1-High: personal outreach", "2-Medium: automated nurture"],
                           "3-Low: business as usual")

tiers = scored.groupby("tier").agg(customers=("churn_risk", "size"), avg_risk=("churn_risk", "mean"),
                                   expected_churners=("churn_risk", "sum"), actual_churn_rate=("actual_churned", "mean"))
tiers["share of all churners"] = scored.groupby("tier")["actual_churned"].sum() / scored["actual_churned"].sum()
tiers["suggested treatment"] = ["Call from a retention rep with a tailored offer",
                                "Email/in-app sequence targeting their risk reasons",
                                "No action; monitor monthly"][:len(tiers)]
display(tiers.style.format({"avg_risk": "{:.1%}", "expected_churners": "{:,.0f}", "actual_churn_rate": "{:.1%}",
                            "share of all churners": "{:.0%}"}))

In [ ]:
# ============================================================
# CALL LIST WITH PER-CUSTOMER REASONS
# A "reason" = a numeric driver where this customer sits in the riskiest 25%
# (direction taken from the logistic model), or a category group whose
# churn rate in the training data is at least 1.25x the average.
# ============================================================
q_hi = df[NUMERIC_COLUMNS].quantile(0.75); q_lo = df[NUMERIC_COLUMNS].quantile(0.25)
train_rows = df.loc[X_train.index]
RISKY_GROUPS = {c: set(train_rows.groupby(c)["_y"].mean().loc[lambda r: r >= 1.25 * CHURN_RATE].index)
                for c in CATEGORICAL_COLUMNS}
print("High-churn groups used as reasons:", {c: sorted(map(str, v)) for c, v in RISKY_GROUPS.items() if v})
driver_order = list(imp.index)            # most important first

def reasons_for(i):
    out = []
    for col in driver_order:
        if col in DIRECTION:
            v = df.at[i, col]
            if pd.isna(v):
                continue
            if (DIRECTION[col] > 0 and v >= q_hi[col]) or (DIRECTION[col] < 0 and v <= q_lo[col]):
                out.append(f"{col}={v:g}")
        elif col in CATEGORICAL_COLUMNS:
            if df.at[i, col] in RISKY_GROUPS[col]:
                out.append(f"{col}={df.at[i, col]}")
        if len(out) == 2:
            break
    return out

call_list = scored[scored["tier"].str.startswith("1")].sort_values("churn_risk", ascending=False).copy()
call_list["top reasons"] = [", ".join(reasons_for(i)) or "overall profile" for i in call_list.index]
call_list["suggested action"] = [DRIVER_ACTIONS.get(r.split("=")[0], "General retention check-in") if r != "overall profile"
                                 else "General retention check-in" for r in call_list["top reasons"]]
call_list["expected value of call"] = call_list["churn_risk"] * VALUE_IF_SAVED * ACTION_SUCCESS_RATE - ACTION_COST
show_cols = ([ID_COLUMN] if ID_COLUMN else []) + ["churn_risk", "top reasons", "suggested action", "expected value of call"]
display(call_list[show_cols].head(15).style.format({"churn_risk": "{:.1%}", "expected value of call": "${:,.0f}"}).hide(axis="index"))
print(f"{len(call_list):,} customers on the call list.")

In [ ]:
# ============================================================
# BUSINESS MEMO (auto-filled; edit before sending)
# ============================================================
driver_lines = "\n".join(f"{k + 1}. **{r['driver']}** ({r['direction']}): {r['recommended action']}."
                         for k, (_, r) in enumerate(TOP_DRIVERS.iterrows()))
high_t = tiers.iloc[0]
memo = f"""
# Churn Risk Program: Recommendation
**{COMPANY_NAME}** · Model: {CHAMPION_NAME} · Customers analyzed: {len(df):,} · Historical churn rate: {CHURN_RATE:.1%}

## Bottom line
Contact the **{int(high_t['customers']):,} highest-risk customers** each cycle. Based on held-out testing, this reaches about
**{TEST_RECALL:.0%} of customers who would churn** with **{TEST_PRECISION:.0%} precision**, for an estimated
**${TEST_VALUE:,.0f} net value per cycle** at our current assumptions (value ${VALUE_IF_SAVED:,} per save,
{ACTION_SUCCESS_RATE:.0%} save rate, ${ACTION_COST} per contact).

## How good is the model?
- It ranks customers well: the riskiest 10% churn at **{TOP10_LIFT:.1f}× the average rate**, and the top 20% contain
  **{TOP20_CAPTURE:.0f}% of all churners** (random selection would find 20%).
- Test-set ROC-AUC {TEST_ROC:.2f}, PR-AUC {TEST_PR:.2f} (random = {y_test.mean():.2f}).
- We chose {CHAMPION_NAME.lower()} because {'it beat the simpler model by more than normal variation' if CHAMPION_NAME == 'Random forest' else 'the more complex forest did not beat it by more than normal variation, and it is easier to explain'}.

## Why customers leave (and what to do)
{driver_lines}

## Program economics
- Break-even: at least **{BREAK_EVEN:.0%}** of contacted customers must be real churn risks. We expect ~{TEST_PRECISION:.0%}.
- Value stays positive in **{POSITIVE_SCENARIOS} of 9** scenarios when save value and save rate are 50% lower or higher than assumed.

## Proposed pilot (before full rollout)
1. Score current customers; take the high-risk tier.
2. Randomly split it: **80% treatment** (outreach) and **20% control** (no outreach).
3. Run for one billing cycle (or 90 days); compare churn between the two groups.
4. Scale if treatment churn is meaningfully lower and the value per contact beats ${ACTION_COST}.

## Caveats
- Drivers are associations from historical data, not proven causes; the pilot tests causality.
- Retrain quarterly, or sooner if pricing, product, or the customer mix changes.
"""
display(Markdown(memo))

## What To Do Next

**This week**
- Review the call list with the retention team; sanity-check a dozen customers against what reps know
- Agree on the treatment for each tier and on the control group (randomly hold out ~20% of the high tier)

**This month: run the pilot**
- Track for both groups: churn rate, offers accepted, revenue retained, cost per contact
- Success = treatment churn meaningfully below control **and** value per contact above `ACTION_COST`

**Ongoing: monitoring and retraining**
- Monthly: average predicted risk vs. actual churn; top-decile churn rate (lift) on new data
- Quarterly: retrain on the latest history, re-run this notebook end to end, compare with the last model
- Alarm bells: input distributions shifting (new plans, pricing changes), lift falling, a new input looking "too good" (leakage)

**Improve the model**
- Add behavior trends (logins this month vs. last month), billing events, and support sentiment
- If the forest won narrowly, try the Gradient Boosting notebook; if logistic won, keep it and invest in better inputs

In [ ]:
# ============================================================
# LEARNING CURVE: would more customer history help?
# Validation still rising on the right = more data likely helps.
# Flat and close to training = invest in better inputs instead.
# ============================================================
sizes, tr_s, va_s = learning_curve(champion, X_train, y_train, cv=OOF_CV, scoring=MAIN_METRIC,
                                   train_sizes=np.linspace(0.2, 1.0, 5), n_jobs=-1)
plt.figure(figsize=(7, 4))
plt.plot(sizes, tr_s.mean(1), "o-", label="Training")
plt.fill_between(sizes, va_s.mean(1) - va_s.std(1), va_s.mean(1) + va_s.std(1), alpha=0.2)
plt.plot(sizes, va_s.mean(1), "o-", label="Validation")
plt.xlabel("Training customers"); plt.ylabel(KEY); plt.title(f"Learning curve: {CHAMPION_NAME}"); plt.legend(); plt.show()
slope = va_s.mean(1)[-1] - va_s.mean(1)[-2]
print("More data still helping." if slope > va_s.std(1)[-1] / 2 else "Curve has flattened: better inputs > more rows.")

## Small Data vs. Large Data

**Small data (under ~200 churners):**
- Repeated stratified CV (automatic); always read results against the ± spread
- Logistic regression usually wins the champion rule, and that's fine: it's explainable and stable
- Keep 5–8 inputs with a clear business story; group rare categories
- Present drivers as ranges and directions, not precise numbers; lean on the pilot for proof

**Large data (thousands of churners and up):**
- The forest (or gradient boosting) is more likely to earn the champion spot
- Tune on a sample to save time (see the Random Forest notebook's `SAMPLE_ROWS_FOR_SEARCH`), then refit on everything
- Use a **time-based** test (train on older months, test on the latest) to mimic real deployment
- Small effects become statistically detectable but may not be worth acting on; judge by dollars, not decimals

In [ ]:
# ============================================================
# SAVE: champion model, memo, and call-list preview
# ============================================================
import joblib, json, datetime

final_champion = clone(champion).fit(X, y)          # refit on ALL labeled customers for future scoring
joblib.dump({"model": final_champion, "champion": CHAMPION_NAME, "threshold": THRESHOLD,
             "features": feature_cols, "trained_on_rows": len(df),
             "trained_at": datetime.date.today().isoformat()}, "churn_champion_model.joblib")

with open("churn_business_memo.md", "w") as f:
    f.write(memo)

preview = call_list[show_cols + ["tier"]].round(3)
preview.to_csv("churn_call_list_preview.csv", index=False)

print("Saved:")
print("  churn_champion_model.joblib   (pipeline + threshold + feature list)")
print("  churn_business_memo.md        (edit and share)")
print(f"  churn_call_list_preview.csv   ({len(preview):,} customers)")
# Next cycle:
#   b = joblib.load("churn_champion_model.joblib")
#   current = pd.read_csv("active_customers.csv")
#   current["churn_risk"] = b["model"].predict_proba(current[b["features"]])[:, 1]
#   to_call = current[current["churn_risk"] >= b["threshold"]].nlargest(TEAM_CAPACITY or len(current), "churn_risk")

## Final Checklist

**Data**
- [ ] Churn definition and scoring date agreed with the business
- [ ] No leakage: every input known **before** the scoring date; sniff test passed
- [ ] Data size checked; small-data path used if churners are few

**Modeling**
- [ ] Both models beat the dummy baseline
- [ ] Logistic tuned with grid search, forest with random search; simplicity rules applied
- [ ] Champion picked by the pre-agreed rule, **before** looking at the test set
- [ ] Test set used once

**Business**
- [ ] Threshold set from out-of-fold expected value and team capacity, not 0.5
- [ ] Sensitivity checked: the program pays off under pessimistic assumptions
- [ ] Top drivers explained in plain language and mapped to actions
- [ ] Risk tiers and call list reviewed with the team
- [ ] Pilot designed with a randomized control group
- [ ] Monitoring and retraining schedule set